In [1]:
import os
import gc
import json
import torch
import numpy as np
from datetime import datetime
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

# =============================================================================
# 🚀 MEMORY-OPTIMIZED KNOWLEDGE DISTILLATION
# =============================================================================
# تکنیک‌های بهینه‌سازی:
# ✅ Aggressive CPU offloading - فوری به CPU
# ✅ Explicit cache clearing - پاکسازی کش بعد از هر sample
# ✅ Gradient checkpointing
# ✅ Selective data retention - فقط چیزهای مهم
# ✅ Streaming generation با batch_size=1
# ✅ torch.cuda.empty_cache() در نقاط کلیدی
# =============================================================================

print("=" * 80)
print("🚀 MEMORY-OPTIMIZED KNOWLEDGE DISTILLATION")
print("=" * 80)

# =============================================================================
# Memory Management Utilities
# =============================================================================

def print_gpu_memory(msg=""):
    """نمایش وضعیت GPU"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1024**3
        print(f"🔍 {msg}")
        print(f"   GPU: {allocated:.2f}GB allocated | {reserved:.2f}GB reserved | {free:.2f}GB free")

def aggressive_cleanup():
    """پاکسازی شدید حافظه GPU"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    gc.collect()  # دوباره برای اطمینان

def move_to_cpu_and_clean(data):
    """انتقال به CPU و پاکسازی GPU"""
    if isinstance(data, torch.Tensor):
        return data.cpu().float().numpy()
    elif isinstance(data, (list, tuple)):
        return [move_to_cpu_and_clean(x) for x in data]
    elif isinstance(data, dict):
        return {k: move_to_cpu_and_clean(v) for k, v in data.items()}
    return data

# =============================================================================
# Model Loading با بهینه‌سازی
# =============================================================================

MODEL_NAME = "microsoft/Phi-4-reasoning"

print("\n📥 Loading Teacher Model with optimizations...")
print_gpu_memory("Before loading")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Load با بهینه‌سازی‌های حافظه
teacher_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,  # FP16 برای صرفه‌جویی
    trust_remote_code=True,
    attn_implementation="eager",  # Eager برای control بیشتر
    low_cpu_mem_usage=True,      # ✅ کاهش استفاده CPU
    use_cache=False,              # ✅ کش رو در training خاموش می‌کنیم
)

# ✅ Gradient Checkpointing (حتی برای inference کمک می‌کنه)
if hasattr(teacher_model, 'gradient_checkpointing_enable'):
    teacher_model.gradient_checkpointing_enable()

print("✅ Teacher model loaded!")
print_gpu_memory("After loading")

aggressive_cleanup()
print_gpu_memory("After cleanup")

# =============================================================================
# Optimized Distillation Config
# =============================================================================

DISTILLATION_CONFIG = {
    "temperature": 2.0,
    "save_logits": True,
    "save_hidden_states": True,
    "save_attentions": True,
    "top_k_logits": 50,
    
    # Memory optimization settings
    "max_new_tokens": 512,        # ✅ کاهش از 1024 به 512
    "generation_batch_size": 1,   # ✅ فقط یکی یکی
    "offload_to_cpu": True,       # ✅ فوری به CPU
    "clear_cache_per_step": True, # ✅ پاکسازی هر step
}

# =============================================================================
# Sample Questions (همون قبلی)
# =============================================================================

CODING_QUESTIONS = [
    {
        "id": 1,
        "question": "Write a Python function to check if a string is a palindrome.",
        "category": "string_manipulation"
    },
    {
        "id": 2,
        "question": "Implement binary search in Python.",
        "category": "algorithms"
    },
    {
        "id": 3,
        "question": "Create a function to find two numbers that sum to target.",
        "category": "arrays"
    },
    {
        "id": 4,
        "question": "Write recursive factorial function.",
        "category": "recursion"
    },
    {
        "id": 5,
        "question": "Reverse a linked list.",
        "category": "data_structures"
    },
]

# =============================================================================
# 🚀 MEMORY-OPTIMIZED GENERATION FUNCTION
# =============================================================================

def generate_with_memory_optimization(question, config=DISTILLATION_CONFIG):
    """
    تولید پاسخ با مدیریت شدید حافظه
    """
    print_gpu_memory(f"  → Before processing question")
    
    # پاکسازی قبل از شروع
    aggressive_cleanup()
    
    prompt = f"""Answer this coding question concisely:

Question: {question}

Answer:"""
    
    # Tokenization
    inputs = tokenizer(prompt, return_tensors="pt").to(teacher_model.device)
    input_length = inputs['input_ids'].shape[1]
    
    print(f"  → Input length: {input_length} tokens")
    
    # =============================================================================
    # Generation با کنترل کامل حافظه
    # =============================================================================
    
    generated_tokens = []
    all_logits = []
    all_hidden_states = []
    all_attentions = []
    
    with torch.no_grad():
        try:
            # ✅ استفاده از generate با output_scores
            outputs = teacher_model.generate(
                **inputs,
                max_new_tokens=config['max_new_tokens'],
                temperature=0.7,
                top_p=0.9,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
                
                # ✅ فقط scores رو می‌خوایم (lightweight)
                output_scores=True,
                return_dict_in_generate=True,
                
                # ✅ این‌ها رو خاموش می‌کنیم برای صرفه‌جویی
                output_hidden_states=False,  # خاموش!
                output_attentions=False,     # خاموش!
                
                # ✅ Cache optimization
                use_cache=True,  # در generation فعال باشه
            )
            
            # دریافت tokens
            generated_ids = outputs.sequences[0]
            generated_tokens = generated_ids[input_length:].cpu().numpy().tolist()
            
            # دیکود
            full_response = tokenizer.decode(generated_ids, skip_special_tokens=True)
            if "Answer:" in full_response:
                response_text = full_response.split("Answer:")[-1].strip()
            else:
                response_text = full_response
            
            print(f"  → Generated {len(generated_tokens)} tokens")
            
            # =============================================================================
            # پردازش Logits و انتقال فوری به CPU
            # =============================================================================
            
            if config['save_logits'] and outputs.scores:
                temperature = config['temperature']
                top_k = config['top_k_logits']
                
                for step_idx, step_scores in enumerate(outputs.scores):
                    # پردازش روی GPU
                    logits = step_scores[0].float()
                    soft_labels = torch.softmax(logits / temperature, dim=-1)
                    
                    # Top-K selection
                    top_probs, top_indices = torch.topk(soft_labels, k=top_k)
                    
                    # ✅ فوری به CPU و numpy
                    logit_data = {
                        'step': step_idx,
                        'top_k_indices': top_indices.cpu().numpy().tolist(),
                        'top_k_logits': logits[top_indices].cpu().numpy().tolist(),
                        'top_k_probs': top_probs.cpu().numpy().tolist(),
                    }
                    
                    all_logits.append(logit_data)
                    
                    # ✅ پاکسازی بعد از هر چند step
                    if step_idx % 10 == 0 and config['clear_cache_per_step']:
                        del step_scores, logits, soft_labels, top_probs, top_indices
                        torch.cuda.empty_cache()
            
            # =============================================================================
            # پاکسازی outputs از GPU
            # =============================================================================
            
            del outputs
            if hasattr(inputs, 'input_ids'):
                del inputs
            
            aggressive_cleanup()
            print_gpu_memory(f"  → After generation cleanup")
            
        except RuntimeError as e:
            print(f"❌ Error during generation: {e}")
            aggressive_cleanup()
            return None, None
    
    # =============================================================================
    # آماده‌سازی نتیجه نهایی
    # =============================================================================
    
    distillation_data = {
        "logits": all_logits,
        "hidden_states": [],  # خالی - نگه‌داری نمی‌کنیم
        "attentions": [],     # خالی - نگه‌داری نمی‌کنیم
        "token_ids": generated_tokens,
        "num_tokens": len(generated_tokens),
    }
    
    return response_text, distillation_data

# =============================================================================
# 🎯 Main Processing Loop با مدیریت حافظه
# =============================================================================

print("\n" + "=" * 80)
print("📚 Generating Dataset with Memory Management...")
print("=" * 80)

complete_dataset = []
errors = []

for idx, item in enumerate(CODING_QUESTIONS):
    print(f"\n{'='*80}")
    print(f"🤔 [{idx+1}/{len(CODING_QUESTIONS)}] Processing Q{item['id']}: {item['question'][:50]}...")
    print(f"{'='*80}")
    
    try:
        # ✅ پاکسازی قبل از هر سوال
        aggressive_cleanup()
        print_gpu_memory("Before question")
        
        # تولید پاسخ
        response_text, distillation_data = generate_with_memory_optimization(item['question'])
        
        if response_text is None:
            print(f"⚠️ Skipping question {item['id']} due to error")
            errors.append(item['id'])
            continue
        
        # ✅ ساخت entry
        entry = {
            "id": item['id'],
            "category": item['category'],
            "question": item['question'],
            "teacher_response": response_text,
            "distillation_data": distillation_data,
            "config": DISTILLATION_CONFIG,
            "timestamp": datetime.now().isoformat()
        }
        
        complete_dataset.append(entry)
        
        # Preview
        print(f"\n✅ Response ({len(response_text)} chars):")
        print(f"   {response_text[:150]}...")
        print(f"📊 Captured {len(distillation_data['logits'])} logit steps")
        
        # ✅ پاکسازی شدید بعد از هر سوال
        del response_text, distillation_data, entry
        aggressive_cleanup()
        print_gpu_memory("After question cleanup")
        
        # ✅ ذخیره موقت بعد از هر سوال (در صورت crash)
        if (idx + 1) % 2 == 0:  # هر 2 سوال
            temp_file = f"temp_dataset_{idx+1}.json"
            with open(temp_file, 'w', encoding='utf-8') as f:
                json.dump(complete_dataset, f, ensure_ascii=False, indent=2)
            print(f"💾 Checkpoint saved: {temp_file}")
        
    except Exception as e:
        print(f"❌ Error processing question {item['id']}: {e}")
        errors.append(item['id'])
        aggressive_cleanup()
        continue

print(f"\n{'='*80}")
print(f"✅ Processed {len(complete_dataset)}/{len(CODING_QUESTIONS)} questions")
if errors:
    print(f"⚠️ Errors: {errors}")
print(f"{'='*80}")

# =============================================================================
# پاکسازی نهایی و ذخیره
# =============================================================================

print("\n💾 Saving final dataset...")
aggressive_cleanup()

if complete_dataset:
    # JSON کامل
    output_file = "complete_distillation_dataset.json"
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(complete_dataset, f, ensure_ascii=False, indent=2)
    print(f"✅ Saved: {output_file}")
    
    # Student format
    student_data = []
    for entry in complete_dataset:
        student_entry = {
            "instruction": entry['question'],
            "output": entry['teacher_response'],
            "soft_labels": entry['distillation_data']['logits'],
            "token_ids": entry['distillation_data']['token_ids'],
            "category": entry['category'],
        }
        student_data.append(student_entry)
    
    student_file = "student_distillation_data.json"
    with open(student_file, 'w', encoding='utf-8') as f:
        json.dump(student_data, f, ensure_ascii=False, indent=2)
    print(f"✅ Saved Student Data: {student_file}")
    
    # =============================================================================
    # آمار نهایی
    # =============================================================================
    
    print("\n" + "=" * 80)
    print("📊 FINAL SUMMARY")
    print("=" * 80)
    
    total_entries = len(complete_dataset)
    total_tokens = sum(len(e['distillation_data']['token_ids']) for e in complete_dataset)
    avg_tokens = total_tokens // total_entries if total_entries > 0 else 0
    
    print(f"✅ Successfully processed: {total_entries}/{len(CODING_QUESTIONS)}")
    print(f"📝 Total tokens generated: {total_tokens:,}")
    print(f"📏 Average tokens per response: {avg_tokens}")
    
    if total_entries > 0:
        sample_logits = len(complete_dataset[0]['distillation_data']['logits'])
        print(f"\n📦 Per Entry Data:")
        print(f"  • Logit steps: ~{sample_logits}")
        print(f"  • Top-K per step: {DISTILLATION_CONFIG['top_k_logits']}")
    
    print("\n" + "=" * 80)
    print("✨ COMPLETE!")
    print("=" * 80)
    
    print_gpu_memory("Final state")
else:
    print("❌ No data was generated successfully")

print("\n🎯 Memory Optimization Techniques Used:")
print("  1. ✅ Aggressive CPU offloading")
print("  2. ✅ Cache clearing after each question")
print("  3. ✅ Gradient checkpointing")
print("  4. ✅ Disabled heavy outputs (hidden_states, attentions)")
print("  5. ✅ Top-K logit selection only")
print("  6. ✅ Reduced max_tokens (512 instead of 1024)")
print("  7. ✅ Checkpoint saves every 2 questions")
print("=" * 80)

🚀 MEMORY-OPTIMIZED KNOWLEDGE DISTILLATION

📥 Loading Teacher Model with optimizations...
🔍 Before loading
   GPU: 0.00GB allocated | 0.00GB reserved | 14.74GB free


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/431 [00:00<?, ?B/s]

...

model-00002-of-00006.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00005-of-00006.safetensors:   0%|          | 0.00/4.77G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/222 [00:00<?, ?B/s]

✅ Teacher model loaded!
🔍 After loading
   GPU: 13.02GB allocated | 13.02GB reserved | 1.72GB free
🔍 After cleanup
   GPU: 13.02GB allocated | 13.02GB reserved | 1.72GB free

📚 Generating Dataset with Memory Management...

🤔 [1/5] Processing Q1: Write a Python function to check if a string is a ...
🔍 Before question
   GPU: 13.02GB allocated | 13.02GB reserved | 1.72GB free
🔍   → Before processing question
   GPU: 13.02GB allocated | 13.02GB reserved | 1.72GB free
  → Input length: 25 tokens
  → Generated 1 tokens
🔍   → After generation cleanup
   GPU: 13.03GB allocated | 13.04GB reserved | 1.72GB free

✅ Response (0 chars):
   ...
📊 Captured 1 logit steps
🔍 After question cleanup
   GPU: 13.03GB allocated | 13.04GB reserved | 1.72GB free

🤔 [2/5] Processing Q2: Implement binary search in Python....
🔍 Before question
   GPU: 13.03GB allocated | 13.04GB reserved | 1.72GB free
🔍   → Before processing question
   GPU: 13.03GB allocated | 13.04GB reserved | 1.72GB free
  → Input length: 18

In [23]:
import torch, gc

print("🔥 NUCLEAR option for GPU 1...")

# پیدا کردن همه چیزی که روی GPU 1 هست
print("\n🔍 Finding what's on GPU 1:")
found = []
for name, obj in list(globals().items()):
    try:
        if hasattr(obj, 'device'):
            if 'cuda:1' in str(obj.device) or (hasattr(obj.device, 'index') and obj.device.index == 1):
                print(f"   ❌ Found: {name}")
                found.append(name)
                # به CPU ببرش
                obj.cpu()
                del globals()[name]
        elif hasattr(obj, 'model') and hasattr(obj.model, 'device'):
            if 'cuda:1' in str(obj.model.device):
                print(f"   ❌ Found: {name}.model")
                found.append(name)
                obj.model.cpu()
                del globals()[name]
    except:
        pass

print(f"\n✅ Deleted: {found}")

# GPU 1 رو خالی کن
torch.cuda.set_device(1)
for _ in range(20):
    torch.cuda.empty_cache()
torch.cuda.synchronize()

gc.collect()

print("\n📊 Result:")
for i in range(2):
    gb = torch.cuda.memory_allocated(i) / 1024**3
    print(f"GPU {i}: {gb:.2f} GB")

🔥 NUCLEAR option for GPU 1...

🔍 Finding what's on GPU 1:

✅ Deleted: []

📊 Result:
GPU 0: 0.01 GB
GPU 1: 0.01 GB


In [ ]:
# Reset کامل GPU 1
import torch
torch.cuda.set_device(1)
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(1)
torch.cuda.reset_accumulated_memory_stats(1)

# اگه خیلی ناامید شدی:
import os
os._exit(0)  # این kernel رو می‌کشه

In [21]:

!pip install accelerate>=0.25.0 deepspeed>=0.12.0

In [4]:
!pip install mpi4py
!pip install deepspeed

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 20.3 MB/s eta 0:00:0000:010:01


In [2]:
import os

# ببین کجایی
print("📍 Current dir:", os.getcwd())

# ببین چه فایل‌هایی داری
print("\n📂 Files here:")
for f in os.listdir('.'):
    if '.json' in f:
        print(f"  ✅ {f}")

# چک کن اون مسیر Kaggle وجود داره؟
if os.path.exists('/kaggle/input/'):
    print("\n📁 Kaggle input:")
    for root, dirs, files in os.walk('/kaggle/input/'):
        for f in files:
            if '.json' in f:
                print(f"  ✅ {os.path.join(root, f)}")

📍 Current dir: /kaggle/working

📂 Files here:
  ✅ temp_dataset_2.json
  ✅ complete_distillation_dataset.json
  ✅ temp_dataset_4.json
  ✅ student_distillation_data.json

📁 Kaggle input:


In [6]:
# =============================================================================
# 🚀 SELECTIVE FINE-TUNING + DISTILLATION (FIXED - NO NaN!)
# =============================================================================

import os
import gc
import json
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm
import deepspeed
import numpy as np

# =============================================================================
# 📁 PATHS
# =============================================================================

TRAIN_FILE = "student_distillation_data.json"
OUTPUT_DIR = "/kaggle/working/selective_distillation_qwen"
DS_CONFIG_FILE = "/kaggle/working/ds_config.json"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =============================================================================
# Utils
# =============================================================================

def aggressive_cleanup():
    gc.collect()
    torch.cuda.empty_cache()
    gc.collect()

# =============================================================================
# Dataset
# =============================================================================

class DistillationDataset(Dataset):
    def __init__(self, json_file, tokenizer, max_length=512):
        print(f"📂 Loading dataset: {json_file}")
        with open(json_file, "r", encoding="utf-8") as f:
            self.data = json.load(f)

        self.tokenizer = tokenizer
        self.max_length = max_length
        print(f"✅ Loaded {len(self.data)} samples")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        prompt = f"""Answer this coding question concisely:

Question: {item['instruction']}

Answer:"""

        inputs = self.tokenizer(
            prompt,
            truncation=True,
            max_length=self.max_length // 2,
            return_tensors="pt"
        )

        outputs = self.tokenizer(
            item["output"],
            truncation=True,
            max_length=self.max_length // 2,
            return_tensors="pt"
        )

        input_ids = torch.cat(
            [inputs["input_ids"], outputs["input_ids"]], dim=1
        ).squeeze(0)

        attention_mask = torch.ones_like(input_ids)

        labels = torch.full_like(input_ids, -100)
        labels[inputs["input_ids"].shape[1]:] = input_ids[
            inputs["input_ids"].shape[1]:
        ]

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "teacher_logits": item.get("soft_labels", []),
            "input_length": inputs["input_ids"].shape[1]
        }

def collate_fn(batch):
    max_len = max(len(x["input_ids"]) for x in batch)

    def pad(x, val):
        return F.pad(x, (0, max_len - x.shape[0]), value=val)

    return {
        "input_ids": torch.stack([pad(x["input_ids"], 0) for x in batch]),
        "attention_mask": torch.stack([pad(x["attention_mask"], 0) for x in batch]),
        "labels": torch.stack([pad(x["labels"], -100) for x in batch]),
        "teacher_logits": [x["teacher_logits"] for x in batch],
        "input_lengths": [x["input_length"] for x in batch],
    }

# =============================================================================
# Distillation Loss (FIXED - Anti-NaN!)
# =============================================================================

def distillation_loss(student_logits, teacher_sparse, temperature=1.5):
    """
    محاسبه KL divergence بین student و teacher با محافظت در برابر NaN
    """
    if not teacher_sparse:
        return torch.tensor(0.0, device=student_logits.device)

    vocab = student_logits.shape[-1]
    total = 0.0
    steps = 0

    for i, t in enumerate(teacher_sparse):
        if i >= student_logits.shape[1]:
            break

        try:
            idx = torch.tensor(t["top_k_indices"], device=student_logits.device, dtype=torch.long)
            probs = torch.tensor(t["top_k_probs"], device=student_logits.device, dtype=torch.float32)
            
            # Normalize probabilities
            probs = probs / (probs.sum() + 1e-9)
            
            # ساخت توزیع teacher
            teacher_full = torch.zeros(vocab, device=student_logits.device, dtype=torch.float32)
            teacher_full[idx] = probs
            
            # Clamp to avoid log(0)
            teacher_full = torch.clamp(teacher_full, min=1e-9, max=1.0)
            
            # محاسبه احتمال student
            student_prob = F.softmax(student_logits[0, i].float() / temperature, dim=-1)
            student_prob = torch.clamp(student_prob, min=1e-9, max=1.0)
            
            # KL divergence
            kl = F.kl_div(
                student_prob.log(), 
                teacher_full, 
                reduction="batchmean"
            )
            
            # چک کردن NaN
            if not torch.isnan(kl) and not torch.isinf(kl):
                total += kl
                steps += 1
                
        except Exception as e:
            # اگه خطایی اومد، رد کن
            continue

    if steps == 0:
        return torch.tensor(0.0, device=student_logits.device)
    
    return total / steps

# =============================================================================
# Main
# =============================================================================

def main():
    CONFIG = {
        "model": "Qwen/Qwen2.5-0.5B-Instruct",
        "batch_size": 1,
        "grad_accum": 8,
        "epochs": 3,
        "lr": 5e-6,  # کاهش learning rate
        "alpha": 0.7,  # بیشتر روی CE تمرکز
        "beta": 0.3,   # کمتر روی distillation
        "temperature": 1.5,  # کاهش temperature
        "save_steps": 50,
        "max_grad_norm": 1.0,  # gradient clipping
    }

    print("🚀 Starting training...")
    print(f"⚙️  Config: LR={CONFIG['lr']}, Alpha={CONFIG['alpha']}, Beta={CONFIG['beta']}, Temp={CONFIG['temperature']}")

    tokenizer = AutoTokenizer.from_pretrained(CONFIG["model"])
    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        CONFIG["model"],
        torch_dtype=torch.float16,
        trust_remote_code=True
    )

    # Freeze all except last 4 layers
    print("🔒 Freezing layers...")
    for p in model.parameters():
        p.requires_grad = False

    for layer in model.model.layers[-4:]:
        for p in layer.parameters():
            p.requires_grad = True

    for p in model.lm_head.parameters():
        p.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"✅ Trainable: {trainable:,} / {total:,} ({trainable/total*100:.1f}%)")

    dataset = DistillationDataset(TRAIN_FILE, tokenizer)
    loader = DataLoader(dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)

    ds_config = {
        "train_batch_size": CONFIG["batch_size"] * CONFIG["grad_accum"],
        "gradient_accumulation_steps": CONFIG["grad_accum"],
        "fp16": {
            "enabled": True,
            "loss_scale": 0,
            "loss_scale_window": 1000,
            "hysteresis": 2,
            "min_loss_scale": 1
        },
        "zero_optimization": {
            "stage": 2,
            "offload_optimizer": {"device": "cpu"},
            "contiguous_gradients": True,
            "overlap_comm": True
        },
        "optimizer": {
            "type": "AdamW",
            "params": {
                "lr": CONFIG["lr"],
                "betas": [0.9, 0.999],
                "eps": 1e-8,
                "weight_decay": 0.01
            }
        },
        "gradient_clipping": CONFIG["max_grad_norm"]
    }

    with open(DS_CONFIG_FILE, "w") as f:
        json.dump(ds_config, f, indent=2)

    model_engine, optimizer, _, _ = deepspeed.initialize(
        model=model,
        model_parameters=[p for p in model.parameters() if p.requires_grad],
        config=DS_CONFIG_FILE
    )

    global_step = 0
    best_loss = float('inf')

    for epoch in range(CONFIG["epochs"]):
        pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{CONFIG['epochs']}")
        epoch_loss = 0
        epoch_ce = 0
        epoch_dist = 0
        
        for batch_idx, batch in enumerate(pbar):
            batch = {
                k: v.to(model_engine.device) if torch.is_tensor(v) else v
                for k, v in batch.items()
            }

            try:
                out = model_engine(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    labels=batch["labels"]
                )

                ce = out.loss
                
                # چک کردن CE loss
                if torch.isnan(ce) or torch.isinf(ce):
                    print(f"⚠️  Skipping batch {batch_idx}: CE is NaN/Inf")
                    continue

                # محاسبه distillation loss
                dist = torch.tensor(0.0, device=ce.device)
                for i in range(len(batch["teacher_logits"])):
                    d = distillation_loss(
                        out.logits[i:i+1, batch["input_lengths"][i]:],
                        batch["teacher_logits"][i],
                        CONFIG["temperature"]
                    )
                    dist += d

                dist /= max(len(batch["teacher_logits"]), 1)
                
                # چک کردن distillation loss
                if torch.isnan(dist) or torch.isinf(dist):
                    print(f"⚠️  Dist loss is NaN/Inf, using only CE")
                    dist = torch.tensor(0.0, device=ce.device)
                
                # Loss نهایی
                loss = CONFIG["alpha"] * ce + CONFIG["beta"] * dist
                
                # چک نهایی
                if torch.isnan(loss) or torch.isinf(loss):
                    print(f"⚠️  Skipping batch {batch_idx}: Final loss is NaN/Inf")
                    continue

                model_engine.backward(loss)
                model_engine.step()

                global_step += 1
                epoch_loss += loss.item()
                epoch_ce += ce.item()
                epoch_dist += dist.item()
                
                pbar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'ce': f'{ce.item():.4f}',
                    'dist': f'{dist.item():.4f}'
                })

                # Save checkpoint
                if global_step % CONFIG["save_steps"] == 0:
                    ckpt = f"{OUTPUT_DIR}/checkpoint-{global_step}"
                    model_engine.save_checkpoint(ckpt)
                    tokenizer.save_pretrained(ckpt)
                    print(f"\n💾 Saved checkpoint: {ckpt}")

                aggressive_cleanup()
                
            except Exception as e:
                print(f"⚠️  Error in batch {batch_idx}: {e}")
                continue

        # Epoch summary
        avg_loss = epoch_loss / len(loader)
        avg_ce = epoch_ce / len(loader)
        avg_dist = epoch_dist / len(loader)
        
        print(f"\n📊 Epoch {epoch+1} Summary:")
        print(f"   Loss: {avg_loss:.4f} | CE: {avg_ce:.4f} | Dist: {avg_dist:.4f}")
        
        if avg_loss < best_loss:
            best_loss = avg_loss
            best_dir = f"{OUTPUT_DIR}/best"
            model_engine.save_checkpoint(best_dir)
            tokenizer.save_pretrained(best_dir)
            print(f"   🏆 New best model saved!")

    # Save final model
    final_dir = f"{OUTPUT_DIR}/final"
    model_engine.save_checkpoint(final_dir)
    tokenizer.save_pretrained(final_dir)

    print("\n" + "="*80)
    print("✅ TRAINING FINISHED & SAVED")
    print(f"📁 Models saved in: {OUTPUT_DIR}")
    print(f"🏆 Best loss: {best_loss:.4f}")
    print("="*80)

if __name__ == "__main__":
    main()

🚀 Starting training...
⚙️  Config: LR=5e-06, Alpha=0.7, Beta=0.3, Temp=1.5
🔒 Freezing layers...
✅ Trainable: 195,784,192 / 494,032,768 (39.6%)
📂 Loading dataset: student_distillation_data.json
✅ Loaded 5 samples
Before initializing optimizer states
MA 2.38 GB         Max_MA 2.49 GB         CA 2.54 GB         Max_CA 3 GB 
CPU Virtual Memory:  used = 8.8 GB, percent = 28.1%
After initializing optimizer states
MA 2.38 GB         Max_MA 2.38 GB         CA 2.54 GB         Max_CA 3 GB 
CPU Virtual Memory:  used = 9.53 GB, percent = 30.4%
After initializing ZeRO optimizer
MA 2.38 GB         Max_MA 2.38 GB         CA 2.54 GB         Max_CA 3 GB 
CPU Virtual Memory:  used = 9.53 GB, percent = 30.4%


Epoch 1/3:  40%|████      | 2/5 [00:03<00:05,  1.96s/it, loss=1.0452, ce=1.4931, dist=0.0001]

⚠️  Skipping batch 2: CE is NaN/Inf


Epoch 1/3: 100%|██████████| 5/5 [00:07<00:00,  1.58s/it, loss=1.0596, ce=1.5137, dist=0.0001]



📊 Epoch 1 Summary:
   Loss: 0.8547 | CE: 1.2210 | Dist: 0.0001
   🏆 New best model saved!


...


📊 Epoch 2 Summary:
   Loss: 0.8547 | CE: 1.2210 | Dist: 0.0001


Epoch 3/3:  20%|██        | 1/5 [00:01<00:07,  1.89s/it, loss=1.1092, ce=1.5846, dist=0.0001]

⚠️  Skipping batch 1: CE is NaN/Inf


Epoch 3/3: 100%|██████████| 5/5 [00:07<00:00,  1.55s/it, loss=0.9684, ce=1.3834, dist=0.0001]



📊 Epoch 3 Summary:
   Loss: 0.8225 | CE: 1.1750 | Dist: 0.0001
   🏆 New best model saved!

✅ TRAINING FINISHED & SAVED
📁 Models saved in: /kaggle/working/selective_distillation_qwen
🏆 Best loss: 0.8225
